# NB02: Gene Content Clustering & COG Profiling

**Goal**: For each target species, cluster genomes by auxiliary gene content to define ecotypes, then compute per-ecotype COG functional profiles.

**Approach**: 
- Stratified sample of 15 species (5 per size bin: 50-100, 100-200, 200-300 genomes)
- Use 2-table join (gene × junction) filtered by genome_id
- Post-filter to auxiliary gene clusters using gene_cluster table
- PCA → KMeans for ecotype clustering (HDBSCAN not available on cluster)
- Batch COG annotation queries (gene_cluster → eggnog_mapper_annotations)

**Execution**: Heavy Spark queries ran via `src/run_clustering.py` standalone script
(~66 min total). This notebook loads and displays the saved results.

**Outputs**: `data/ecotype_assignments.csv`, `data/ecotype_cog_profiles.csv`, `data/clustering_stats.csv`

In [1]:
import pandas as pd
import numpy as np

# Load pre-computed results (generated by src/run_clustering.py)
ecotypes = pd.read_csv('../data/ecotype_assignments.csv')
stats = pd.read_csv('../data/clustering_stats.csv')
cog_profiles = pd.read_csv('../data/ecotype_cog_profiles.csv')

print(f"Ecotype assignments: {len(ecotypes)} genomes across {ecotypes['species'].nunique()} species")
print(f"COG profiles: {len(cog_profiles)} entries across {cog_profiles['species'].nunique()} species")
print(f"Clustering stats: {len(stats)} species")

Ecotype assignments: 1820 genomes across 12 species
COG profiles: 894 entries across 12 species
Clustering stats: 12 species


## 1. Clustering Summary

In [2]:
print(f"=== Clustering Summary ===")
print(f"Species sampled: 15 (stratified: 5 per size bin)")
print(f"  50-100 genomes: 5 species")
print(f"  100-200 genomes: 5 species")
print(f"  200-300 genomes: 5 species")
print(f"\nSpecies with valid ecotypes: {len(stats)}/15")
print(f"  (2 species had S3 read errors, 1 had too few genomes for clustering)")
print(f"\nTotal genome-ecotype assignments: {len(ecotypes)}")
print(f"Mean silhouette: {stats['silhouette'].mean():.3f}")
print(f"Median silhouette: {stats['silhouette'].median():.3f}")
print(f"Mean clusters/species: {stats['n_clusters'].mean():.1f}")
print(f"\nPer-species breakdown:")
print(f"{'Species':<40} {'Genomes':>8} {'Clusters':>9} {'Silhouette':>11} {'Assigned':>9}")
print("-" * 80)
for _, row in stats.iterrows():
    print(f"{row['short_name']:<40} {row['n_genomes']:>8} {row['n_clusters']:>9} "
          f"{row['silhouette']:>11.3f} {row['n_assigned']:>9}")

=== Clustering Summary ===
Species sampled: 15 (stratified: 5 per size bin)
  50-100 genomes: 5 species
  100-200 genomes: 5 species
  200-300 genomes: 5 species

Species with valid ecotypes: 12/15
  (2 species had S3 read errors, 1 had too few genomes for clustering)

Total genome-ecotype assignments: 1820
Mean silhouette: 0.215
Median silhouette: 0.174
Mean clusters/species: 3.7

Per-species breakdown:
Species                                   Genomes  Clusters  Silhouette  Assigned
--------------------------------------------------------------------------------
Staphylococcus_simulans                        78         3       0.118        63
Enterococcus_D_gallinarum                      95         3       0.245        86
Pectobacterium_carotovorum                     57         2       0.196        57
Streptococcus_pseudopneumoniae                127         4       0.131       118
Bacillus_safensis                             120         3       0.141       117
Limosilactobacillus

## 2. COG Profile Overview

In [3]:
print(f"COG profile entries: {len(cog_profiles)}")
print(f"Species with profiles: {cog_profiles['species'].nunique()}")
print(f"COG categories observed: {sorted(cog_profiles['COG_category'].unique())}")
print(f"Unique ecotypes with profiles: {cog_profiles.groupby('species')['ecotype'].nunique().sum()}")

# Show COG category distribution across all species
cog_totals = cog_profiles.groupby('COG_category')['count'].sum().sort_values(ascending=False)
print(f"\nTop 10 COG categories by total gene cluster count:")
for cat, cnt in cog_totals.head(10).items():
    print(f"  {cat}: {cnt:,}")

COG profile entries: 894
Species with profiles: 12
COG categories observed: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'S', 'T', 'U', 'V', 'W', 'Z']
Unique ecotypes with profiles: 44

Top 10 COG categories by total gene cluster count:
  S: 338,157
  L: 121,519
  K: 95,480
  G: 81,158
  M: 80,750
  P: 65,908
  Q: 41,415
  T: 39,594
  E: 39,195
  C: 36,899
